# 01 Raw Data Inspection

Purpose: inspect the raw KKBox files before modeling. This notebook checks target quality, member data quality, transaction coverage, date ranges, and leakage risks.

The goal is not to model yet. The goal is to understand what data is safe and useful.

## Setup

Define project paths once. This avoids repeating long Windows paths and makes the notebook easier to move into scripts later.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_DIR = Path(r"C:\Users\muham\OneDrive\Desktop\ML Journey\Project\kkbox-churn")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

TRAIN_PATH = RAW_DIR / "train_v2.csv"
MEMBERS_PATH = RAW_DIR / "members_v3.csv"
TRANSACTIONS_PATH = RAW_DIR / "transactions.csv"
CUTOFF_DATE = pd.Timestamp("2017-02-28")

for path in RAW_DIR.iterdir():
    size_mb = path.stat().st_size / 1024**2
    print(f"{path.name}: {size_mb:,.2f} MB")

## Label Table Inspection

Start with `train_v2.csv` because it contains the target. If the label table has duplicates, missing values, or an unexpected target rate, everything after it becomes unreliable.

In [ ]:
train = pd.read_csv(TRAIN_PATH)

print("Shape:", train.shape)
print("Columns:", train.columns.tolist())
display(train.head())

print("Target counts:")
print(train["is_churn"].value_counts())

print("Target rate:")
print(train["is_churn"].value_counts(normalize=True).round(4))

print("Missing msno:", train["msno"].isna().sum())
print("Missing is_churn:", train["is_churn"].isna().sum())
print("Duplicate msno:", train["msno"].duplicated().sum())
print("Unique users:", train["msno"].nunique())

### Interpretation

The churn rate is about 8.99%, so this is an imbalanced classification problem. Accuracy is not enough because predicting every user as non-churn would already get about 91% accuracy while catching no churners.

## Member Data Inspection

Member data may help explain churn, but it is incomplete and dirty. We inspect missing values, duplicate IDs, age quality, and member coverage before using it.

In [ ]:
members = pd.read_csv(MEMBERS_PATH)

print("Shape:", members.shape)
print("Columns:", members.columns.tolist())
print("Dtypes:")
print(members.dtypes)
display(members.head())

print("Missing values:")
print(members.isna().sum())
print("Duplicate msno:", members["msno"].duplicated().sum())

member_users = set(members["msno"])
users_with_member_data = train["msno"].isin(member_users).sum()
print("Train users with member data:", users_with_member_data)
print("Train users without member data:", len(train) - users_with_member_data)
print("Coverage rate:", round(users_with_member_data / len(train), 4))

### Interpretation

Most labeled users have member data, but not all. Use a left join from labels to member features and create a `has_member_data` flag instead of dropping users.

## Profile Feature Quality

`bd` means age, not birthday. It has invalid values. `city` and `registered_via` are integer-coded categories, not true numeric measurements.

In [ ]:
print("bd summary:")
print(members["bd"].describe())
print("Smallest bd:", members["bd"].sort_values().head(20).tolist())
print("Largest bd:", members["bd"].sort_values(ascending=False).head(20).tolist())
print("Most common bd:")
print(members["bd"].value_counts().head(20))

print("Gender counts including missing:")
print(members["gender"].value_counts(dropna=False))

print("City counts:")
print(members["city"].value_counts().head(15))

print("Registered_via counts:")
print(members["registered_via"].value_counts().sort_index())

### Interpretation

Treat valid age as 13 to 80. Treat `bd = 0` and impossible ages as missing. Keep an age-missing flag because missing profile data may itself be predictive.

## Registration Date Leakage Check

Registration date can create account tenure, but dates after the cutoff are future information. Future registration-derived values must not be used directly.

In [ ]:
members["registration_init_time_parsed"] = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce",
)

print("Earliest registration:", members["registration_init_time_parsed"].min())
print("Latest registration:", members["registration_init_time_parsed"].max())

train_members = train.merge(
    members[["msno", "registration_init_time", "registration_init_time_parsed"]],
    on="msno",
    how="left",
)
future_reg = train_members["registration_init_time_parsed"] > CUTOFF_DATE
print("Train users with registration after cutoff:", future_reg.sum())

## Transaction Data Inspection

Transactions are the main behavior data. The file is large, so we inspect selected columns first. `transactions.csv` ends at 2017-02-28, which matches the v1 cutoff.

In [ ]:
transactions_sample = pd.read_csv(TRANSACTIONS_PATH, nrows=5)
print("Columns:", transactions_sample.columns.tolist())
print(transactions_sample.dtypes)
display(transactions_sample)

date_check = pd.read_csv(TRANSACTIONS_PATH, usecols=["transaction_date", "membership_expire_date"])
date_check["transaction_date_parsed"] = pd.to_datetime(date_check["transaction_date"].astype(str), format="%Y%m%d", errors="coerce")
date_check["membership_expire_date_parsed"] = pd.to_datetime(date_check["membership_expire_date"].astype(str), format="%Y%m%d", errors="coerce")

print("Rows:", len(date_check))
print("Transaction date range:", date_check["transaction_date_parsed"].min(), date_check["transaction_date_parsed"].max())
print("Membership expire date range:", date_check["membership_expire_date_parsed"].min(), date_check["membership_expire_date_parsed"].max())

### Interpretation

Transaction data is safe for v1 because the maximum transaction date is 2017-02-28. Suspicious expiration dates exist and should be documented as a limitation.